In [ ]:
!pip install -q --no-deps "huggingface-hub==0.36.2" "transformers==4.48.3" "tokenizers>=0.21,<0.22"

In [ ]:
B64 = '...'
print(len(B64), 'base64 chars')

In [ ]:
import base64, gzip, io, os, time, gc, json
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| transformers 4.48.3 | dtype fp32 | batch 8 (last position)')
df = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(B64))), encoding='utf-8')
print(len(df), 'probes')
OUT = '/kaggle/working'
MODELS = [
    ('LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', '8e6fc27d1910b526b5d48a2aa129b08a0293df5e', 'exaone'),
    ('kakaocorp/kanana-1.5-2.1b-instruct-2505', None, 'kanana'),
]
BS = 8
CKPT = 500

def wrap(tok, user):
    if getattr(tok, 'chat_template', None):
        try:
            return tok.apply_chat_template([{'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True) + '정답: '
        except Exception:
            pass
    return user + '\n\n정답: '

def last_logits(model, input_ids, mask, pos):
    base = getattr(model, 'model', None) or getattr(model, 'transformer')
    h = base(input_ids=input_ids, attention_mask=mask, position_ids=pos).last_hidden_state
    return model.lm_head(h[:, -1])

def digit_ids(tok):
    d = {}
    for k in '123456789':
        ids = tok.encode(k, add_special_tokens=False)
        if len(ids) == 1:
            d[k] = ids[0]
    return d

for mid, rev, tag in MODELS:
    path = f'{OUT}/logit_{tag}_fp32.csv'
    done = set(pd.read_csv(path, encoding='utf-8-sig').probe_id) if os.path.exists(path) else set()
    todo = df[~df.probe_id.isin(done)].reset_index(drop=True)
    print(f'\n=== {tag}: 완료 {len(done)}건, 남은 {len(todo)}건 ===', flush=True)
    if len(todo) == 0:
        continue
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True, revision=rev)
    model = AutoModelForCausalLM.from_pretrained(mid, trust_remote_code=True, revision=rev,
                                                 torch_dtype=torch.float32, attn_implementation='eager').to('cuda').eval()
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    digit = digit_ids(tok)
    rows, t0, n = [], time.time(), 0
    for i in range(0, len(todo), BS):
        chunk = todo.iloc[i:i + BS]
        enc = tok([wrap(tok, p) for p in chunk.prompt], return_tensors='pt', padding=True).to('cuda')
        mask = enc['attention_mask']
        pos = (mask.cumsum(dim=1) - 1).clamp(min=0)
        with torch.no_grad():
            lgs = last_logits(model, enc['input_ids'], mask, pos)
        for j, (pid, codes) in enumerate(zip(chunk.probe_id, chunk.codes)):
            p = torch.softmax(lgs[j].float(), dim=-1)
            raw = [p[digit[str(c)]].item() for c in json.loads(codes)]
            m = sum(raw)
            rows.append(dict(probe_id=pid, model=mid, model_dist=json.dumps([x / m for x in raw]), mass=m))
        n += len(chunk)
        if len(rows) >= CKPT or i + BS >= len(todo):
            pd.DataFrame(rows).to_csv(path, mode='a', header=not os.path.exists(path), index=False, encoding='utf-8-sig')
            rows = []
            el = time.time() - t0
            print(f'{tag} {n}/{len(todo)} 경과 {el/60:.1f}분, 남은 예상 {el/n*(len(todo)-n)/60:.1f}분', flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()

for tag in ['exaone', 'kanana']:
    p = f'{OUT}/logit_{tag}_fp32.csv'
    if os.path.exists(p):
        print(tag, len(pd.read_csv(p, encoding='utf-8-sig')), '행')
